In [ ]:
%matplotlib ipympl
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation

import pandas.plotting._matplotlib.converter as pdtc

In [ ]:
geo311 = gpd.read_file("data/geolocated311.json")

In [ ]:
geo311_agg = geo311.pivot_table(values='Additional Details', index='Borough', columns=['Problem Detail (formerly Descriptor)'], aggfunc='count').T
geo311_top5_complaints = geo311_agg.sum(axis=1).sort_values(ascending=False).index[:5]
geo311_top5 = geo311[geo311['Problem Detail (formerly Descriptor)'].isin(geo311_top5_complaints)]

In [ ]:
cdict = dict(zip(geo311['Borough'].unique(), ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e','#e6ab02']))

In [ ]:
fig, axd = plt.subplot_mosaic([[g] for g in geo311_top5_complaints], sharex=True, sharey=True, figsize=(5,5), layout='compressed')

for name, gdfsub in geo311_top5.groupby(['Problem Detail (formerly Descriptor)']):
    axn = name[0]
    axd[axn].set_title(axn)
    for (b, gdfts) in gdfsub.groupby(['Borough']):
        ts = gdfts.groupby(pd.Grouper(key='Created Date', freq='D'))['Problem Detail (formerly Descriptor)'].count()
        l, = axd[axn].plot(ts.index, ts.values, label=b[0], color=cdict[b[0]])
handles, labels = axd[axn].get_legend_handles_labels()
axd[axn].set(xlim=(geo311_top5['Created Date'].min(),geo311_top5['Created Date'].max()))
fig.legend(handles, labels, bbox_to_anchor=(0., 1.02, 1., .102), loc='lower left', ncols=3, mode="expand", borderaxespad=0.)


In [ ]:
fig, axd = plt.subplot_mosaic([[g, f'map_{g}'] for g in geo311_top5_complaints[:1]], figsize=(10,5), layout='compressed', width_ratios=[2,1])

maps = dict()
for name, gdfsub in geo311_top5.groupby(['Problem Detail (formerly Descriptor)']):
    axn = name[0]
    if axn not in axd:
        continue
 
    axd[axn].set_title(axn)
    axd[f'map_{axn}'].set_title(axn)
    for (b, gdfts) in gdfsub.groupby(['Borough']):
        ts = gdfts.groupby(pd.Grouper(key='Created Date', freq='D'))['Problem Detail (formerly Descriptor)'].count()
        l, = axd[axn].plot(ts.index, ts.values, label=b[0], color=cdict[b[0]])
    handles, labels = axd[axn].get_legend_handles_labels()
    if "map" not in axn:
        axd[axn].set(xlim=(geo311_top5['Created Date'].min(),geo311_top5['Created Date'].max()))
    geo311[geo311['Problem Detail (formerly Descriptor)'].str.startswith(axn)].dissolve(by='ct2020', aggfunc='count').plot(column='Unique Key', cmap='YlOrRd', legend=True, ax=axd[f'map_{axn}'], edgecolor='.9', legend_kwds={'shrink':1})

fig.legend(handles, labels, bbox_to_anchor=(0., 1.02, 1., .102), loc='lower left', ncols=3, mode="expand", borderaxespad=0.)
plt.show()


In [ ]:
geo311[geo311['Problem Detail (formerly Descriptor)'].str.startswith("Vendor")].dissolve(by='ct2020', aggfunc='count').plot(column='Unique Key', cmap='viridis', legend=True, ax=axd[f'map_{axn}'], edgecolor='.9', legend_kwds={'shrink':1})

In [ ]:
fig, axd = plt.subplot_mosaic([[g, f'map_{g}'] for g in geo311_top5_complaints], figsize=(10,10), layout='compressed', width_ratios=[2,1])

fig, axd = plt.subplot_mosaic([[g, f'map_{g}'] for g in geo311_top5_complaints[:1]], figsize=(10,5), layout='compressed', width_ratios=[2,1])

maps = dict()
for name, gdfsub in geo311_top5.groupby(['Problem Detail (formerly Descriptor)']):
    axn = name[0]
    if axn not in axd:
        continue
 
    axd[axn].set_title(axn)
    axd[f'map_{axn}'].set_title(axn)
    for (b, gdfts) in gdfsub.groupby(['Borough']):
        ts = gdfts.groupby(pd.Grouper(key='Created Date', freq='D'))['Problem Detail (formerly Descriptor)'].count()
        l, = axd[axn].plot(ts.index, ts.values, label=b[0], color=cdict[b[0]])
    handles, labels = axd[axn].get_legend_handles_labels()
    if "map" not in axn:
        axd[axn].set(xlim=(geo311_top5['Created Date'].min(),geo311_top5['Created Date'].max()))
    geo311[geo311['Problem Detail (formerly Descriptor)'].str.startswith(axn)].dissolve(by='ct2020', aggfunc='count').plot(column='Unique Key', cmap='YlOrRd', legend=True, ax=axd[f'map_{axn}'], edgecolor='.9', legend_kwds={'shrink':1})

fig.legend(handles, labels, bbox_to_anchor=(0., 1.02, 1., .102), loc='lower left', ncols=3, mode="expand", borderaxespad=0.)

def on_xlims_change(axes):
    
    start, end = axd.get_xlim()
    print(start, end)
    sub =  geo311[(start <= geo311['Created Date'] & geo311['Created Date'] <= end) & geo311['Problem Detail (formerly Descriptor)'].str.startswith(axn)].dissolve(by='ct2020', aggfunc='count').plot(column='Unique Key', cmap='YlOrRd', legend=True, ax=axd[f'map_{axn}'], edgecolor='.9', legend_kwds={'shrink':1})
    ax.figure.canvas.draw_idle()

axd['map_Retail Store'].callbacks.connect('xlim_changed', on_xlims_change)
plt.show()
